# 02 — Train M0 (baseline) then M1 (curriculum)
M0 must finish first: M1's config asserts against M0's `run_meta.json` (checkpoint hash + step count). Debug on the 0.5B config before spending T4/L4 hours on 1.5B/3B (Part 2 practical plan).

In [ ]:

# --- Self-contained Colab bootstrap (Part 0) ---
# Every notebook does this independently: Colab does not guarantee a new
# notebook tab reuses a previous notebook's VM, so nothing installed or
# cloned in another notebook can be assumed to exist here. This is
# idempotent -- re-running it (e.g. because you ARE still on the same
# runtime) just no-ops the clone and re-pulls latest.
import os, subprocess, shutil
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

# artifacts/corpus.jsonl and data/eval/*.jsonl are built once in
# 01_data.ipynb but land on THAT run's local (ephemeral) VM disk -- a
# different notebook tab is not guaranteed to reuse the same VM, so
# without this they'd be missing here (this is exactly the
# FileNotFoundError: 'artifacts/corpus.jsonl' failure mode). 01_data.ipynb
# copies them to this same Drive folder after building them; restore them
# here if this fresh VM doesn't have them locally yet. No-ops harmlessly
# if the local copy already exists or Drive doesn't have one yet.
DATA = '/content/drive/MyDrive/verilog-slm/data'
os.makedirs(DATA, exist_ok=True)
os.makedirs('data/eval', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
for rel_path in ['artifacts/corpus.jsonl', 'data/eval/verilogeval_v2.jsonl', 'data/eval/rtllm_v2.jsonl']:
    drive_path = f"{DATA}/{os.path.basename(rel_path)}"
    if os.path.exists(drive_path) and not os.path.exists(rel_path):
        shutil.copy(drive_path, rel_path)
        print(f"restored {rel_path} from Drive")

In [ ]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# requirements-train.txt deliberately doesn't pin torch (Colab ships one
# already matched to the VM's CUDA driver) -- log what's actually here
# instead, per Part 0's "pin every dependency" hygiene.
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())

In [ ]:
import os
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

# verible's release asset filename embeds a version string that changes
# every release (verible-v0.0-NNNN-gHASH-linux-static-x86_64.tar.gz), so a
# fixed "latest/download/<literal-name>" URL goes stale -- resolve the
# actual asset URL via the GitHub API instead. Chained as one shell
# command (not separate `!` lines) so the VERIBLE_URL variable survives
# across the pipe/curl/tar steps -- each `!` line is its own subprocess,
# so a bare shell variable assignment on its own line would silently be
# treated as Python and never reach bash at all.
!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

In [ ]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

## Debug pass (0.5B) -- minutes, not hours
Smoke-tests the actual training code path (QLoRA load, LoRA attach, optimizer/scheduler, batch construction, checkpoint save) end to end before committing real GPU-hours to M0. Uses `configs/debug_0.5b.yaml` -- a separate config (not a hand-edited copy of the real one) so this run can never be mistaken for M0 or accidentally feed M1's `assert_against`.

In [ ]:
!python -m src.train.sft --config configs/debug_0.5b.yaml

In [ ]:
# Sanity check: confirm it actually trained (loss logged, checkpoint saved,
# run_meta written) rather than silently no-oping somewhere.
import json
meta = json.load(open('artifacts/debug_0.5b/run_meta.json'))
print('completed steps:', meta['total_steps'])
print('train GPU-hours:', meta['train_gpu_hours'])
print('checkpoint exists:', __import__('os').path.exists('artifacts/debug_0.5b/final'))
log_lines = open('artifacts/debug_0.5b/train_log.jsonl').readlines()
print('loss log entries:', len(log_lines))
print('first logged loss:', json.loads(log_lines[0])['loss'])
print('last logged loss:', json.loads(log_lines[-1])['loss'])

## Memory-fit check on the REAL M0 model (1.5B, real seq_len=2048)
The 0.5B debug pass above only proved the code path works -- it used a much shorter seq_len to guarantee it fit, so it does NOT prove `base.yaml`'s revised batch/seq_len settings actually fit the real 1.5B model. This runs 5 real steps on the real model to confirm before committing to the full multi-hour run.

In [ ]:
!python -m src.train.sft --config configs/m0_memcheck.yaml

### If the check above passed but was very slow (per-step time implies the full run would take dozens of hours), test a less conservative batch size before committing -- `per_device_batch_size=1` avoids OOM but is GPU-inefficient (one example at a time).

In [ ]:
!python -m src.train.sft --config configs/m0_memcheck_bs2.yaml

## M0 -- flat SFT baseline
The real run. Record: total optimizer steps, wall-clock, GPU model, peak VRAM (Part 5).

In [ ]:
!python -m src.train.sft --config configs/m0_baseline.yaml 2>&1 | tee artifacts_drive_logs/m0_stdout.log
!cp -r artifacts/m0 artifacts_drive_ckpt/m0

## Diagnostic pass (Part 6) -- produces the table M1's reweighting needs
Run before M1. See notebooks/03_diagnose.ipynb for the full breakdown; the minimum needed here is `artifacts/m0_diagnostic.json`.

In [ ]:
!python -m src.infer.generate --adapter artifacts/m0/final --split probe \
  --n 5 --temperature 0.8 --top_p 0.95 --out artifacts/m0_probe_gens.jsonl
!python -m src.eval.diagnose --gens artifacts/m0_probe_gens.jsonl --out artifacts/m0_diagnostic.json

## M1 -- curriculum SFT, same step budget as M0
`assert_matches_m0` inside sft.py fails loudly if the base checkpoint or step count diverge from M0 -- this is the guarantee that keeps the ablation clean (Part 8).

In [ ]:
!python -m src.train.sft --config configs/m1_curriculum.yaml 2>&1 | tee artifacts_drive_logs/m1_stdout.log
!cp -r artifacts/m1 artifacts_drive_ckpt/m1

In [ ]:
# Plot realised category histogram: M1 actually saw vs. M0 (uniform) --
# the direct evidence the curriculum did what it was designed to do (Part 7).
import json, matplotlib.pyplot as plt
m1_meta = json.load(open('artifacts/m1/run_meta.json'))
hist = m1_meta['realised_histogram']['construct']
plt.bar(hist.keys(), hist.values())
plt.xticks(rotation=60, ha='right')
plt.title('M1 realised construct-tag exposure over training')
plt.tight_layout()
plt.savefig('artifacts/m1_realised_histogram.png')
plt.show()